# Snowshoe hares at Bonanza Creek Experimental Forest
## Week 3 - Discussion section

### 1. Archive Exploration

Take some time to look through the dataset’s description in EDI and click around. Then click on View Full Metadata, go to the “Detailed Metadata” section and click on “Data Entities” to look at the descriptions for the dataset’s columns.

Use the dataset’s description and the metadata to answer the following questions:

Who assembled this data, and why?

- Scientists from Bonanza Creek LTER - see citation below for names

What does a single row in the data represent?

- A single row represents a single hare capture event.

When and where were the observations collected?

- From 1999-06-01 to 2012-09-14 (publication in 2002)

Does the dataset contain sensitive data?

- No sensitive data.

What study protocols were used to collect the data?

- 'Population', 'Collar', 'Metabolic', 'Metabolic/Collar' (and NaN)

What are the units for the hares’ weight?

- Grams (g)

Does the metadata say how missing data is encoded?

- Not clearly, but from the study protocol question w/ NaN (at least in the study column)

Is there a publication associated with this dataset?

- Yes, the 2017 citation below and a M.S. thesis in 2002 (Flora, B.K. 2002. Comparison of snowshoe hare populations in Interior Alaska. M.S. Thesis. University of Alaska-Fairbanks.)

Citation: K. Kielland, F. S. Chapin, R. W. Ruess, and Bonanza Creek LTER, “Snowshoe hare physical data in Bonanza Creek Experimental Forest: 1999-Present.” Environmental Data Initiative, 2017. doi: 10.6073/PASTA/03DCE4856D79B91557D8E6CE2CBCDC14. Available: https://portal.edirepository.org/nis/mapbrowse?packageid=knb-lter-bnz.55.22. [Accessed: Oct. 14, 2026]

Questions about possible issues in the data:

1. Are missing values NaN in all columns?

2. This is a wide time range - is data collection terminology consistent?

### 2. Adding an Image

![(Brown) Snowshoe hare image](https://thumb.wikimedia.org/wikipedia/commons/thumb/8/8a/SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg/1920px-SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20170313021652)

### 3. Data Loading and Preliminary Exploration

What are the dimensions of the data frame and what are the data types of the columns? Do the data types match what you would expect from each column?

Are there any columns that have a significant number of NA values?

What are the minimum and maximum values for the weight and hind feet measurements?

What are the unique values for some of the categorical columns?

Did any of the issues you anticipated in 1c show up?

In [45]:
import pandas as pd

# Load data
hares = pd.read_csv('data/55_Hare_Data_2012.txt')

In [46]:
# Types of studies performed
hares['study'].unique()

array(['Population', 'Collar', nan, 'Metabolic', 'Metabolic/Collar'],
      dtype=object)

In [47]:
# Dimensions of hares (row, columns)
hares.shape

(3380, 14)

In [ ]:
# Column data types in hares
hares.info()

# hindft is a float64 despite being listed in the metadata as an integer
# b_key and session_id should both be integers according to the metadata, but b_key is a float64
# age is a string, which matches the metadata, but seems odd
# The following columns have more than 100 NaN values: time, r_ear, sex, age, weight, hindft, notes, study

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3380 entries, 0 to 3379
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        3380 non-null   object 
 1   time        264 non-null    object 
 2   grid        3380 non-null   object 
 3   trap        3368 non-null   object 
 4   l_ear       3332 non-null   object 
 5   r_ear       3211 non-null   object 
 6   sex         3028 non-null   object 
 7   age         1269 non-null   object 
 8   weight      2845 non-null   float64
 9   hindft      1633 non-null   float64
 10  notes       243 non-null    object 
 11  b_key       3333 non-null   float64
 12  session_id  3380 non-null   int64  
 13  study       3217 non-null   object 
dtypes: float64(3), int64(1), object(10)
memory usage: 369.8+ KB


In [48]:
# Min and max for weight
print("Min weight:", hares['weight'].min())
print("Max weight:", hares['weight'].max())

Min weight: 0.0
Max weight: 2365.0


In [49]:
# Min and max for hindft
print("Min hindft:", hares['hindft'].min())
print("Max hindft:", hares['hindft'].max())

Min hindft: 60.0
Max hindft: 160.0


In [50]:
# Unique values for some categorical columns
print("Unique sex values:", hares['sex'].unique())
print("Unique age values:", hares['age'].unique())
print("Unique study values:", hares['study'].unique())


Unique sex values: [nan 'M' 'F' '?' 'F?' 'M?' 'pf' 'm' 'f' 'f?' 'm?' 'f ' 'm ']
Unique age values: [nan 'J' 'A' 'a 1 yr.' 'a 3/4 yr.' 'a 1 yr' '1 yr' '1 yr.' '2 yrs.'
 '2 yrs' 'a 2 yrs.' '2.25 yrs' '3.5 yrs.' '3 yrs.' '2.5 yrs' '3.25 yrs.'
 'A 1.5' '?' 'U' 'j' 'a' 'u' 'J 3/4' 'A 3/4' 'A 1/2' '3/4/2013' '1/4/2013'
 '1/2/2013' '1' '1.25' '1.5']
Unique study values: ['Population' 'Collar' nan 'Metabolic' 'Metabolic/Collar']


Data terminology - particularly in string/object columns - does vary, possibly by year of record but also likely by observer. The sex and age columns in particular will require extreme data cleaning.

### 4. Detecting messy values

In the metadata section of the EDI repository, find the allowed values for the hares’ sex. Create a small table in a markdown cell showing the values and their definitions.

Get the number of times each unique sex non-NA value appears.

Check the documentation of value_counts(). What is the purpose of the dropna parameter and what is its default value? Repeat step (b), this time adding the dropna=False parameter to value_counts().

Discuss with your team the output of the unique value counts. In particular:

Do the values in the sex column correspond to the values declared in the metadata?
What could have been potential causes for multiple codes?
Are there seemingly repeated values? If so, what could be the cause?
Write code to confirm your suspicions about d-iii.

| Sex Value | Description |
| --------- | ---------- |
| m | Male |
| f | Female |
| m? | male not confirmed |

In [51]:
# Count of each sex value (not incl. NAs)
hares['sex'].value_counts()

sex
F     1161
M      730
f      556
m      515
?       40
F?      10
f        4
m        4
f?       3
M?       2
m?       2
pf       1
Name: count, dtype: int64

In [35]:
# Count of each sex value (w/ dropna = False)
hares['sex'].value_counts(dropna=False)

sex
F      1161
M       730
f       556
m       515
NaN     352
?        40
F?       10
f         4
m         4
f?        3
M?        2
m?        2
pf        1
Name: count, dtype: int64

In [52]:
# Remove spaces, change all to lowercase to reduce apparently repetetive values
hares['sex_simple'] = hares['sex'].str.strip().str.lower()#.value_counts(dropna = False)

# Change hares df
#hares['sex'] = hares['sex'].str.strip().str.lower()

hares.head()

,date,time,grid,trap,l_ear,r_ear,sex,age,weight,hindft,notes,b_key,session_id,study,sex_simple
0,11/26/1998,NaN,bonrip,1A,414D096A08,NaN,NaN,NaN,1370.0,160.0,NaN,917.0,51,Population,NaN
1,11/26/1998,NaN,bonrip,2C,414D320671,NaN,M,NaN,1430.0,NaN,NaN,936.0,51,Population,m
2,11/26/1998,NaN,bonrip,2D,414D103E3A,NaN,M,NaN,1430.0,NaN,NaN,921.0,51,Population,m
3,11/26/1998,NaN,bonrip,2E,414D262D43,NaN,NaN,NaN,1490.0,135.0,NaN,931.0,51,Population,NaN
4,11/26/1998,NaN,bonrip,3B,414D2B4B58,NaN,NaN,NaN,1710.0,150.0,NaN,933.0,51,Population,NaN


### 5. Brainstorm

Individually, write step-by-step instructions on how you would wrangle the hares data frame to clean the values in the sex column to have only two classes female and male. Which codes would you assign to each new class? Remember: It’s ok if you don’t know how to code each step - it’s more important to have an idea of what you would like to do.

Discuss your step-by-step instructions with your team.

The next exercise asks you to clean the sex codes. There are many ways of doing this. Use the approach you came up with in your brainstorm, or try a different one!

- Step 1: change all observation to lowercase
- Step 2: remove spaces (strip)
- Step 3: convert f? and ? to m?

In [ ]:
# Change sex values not represented in study design to m?
hares.loc[(hares['sex_simple'] == 'f?') | (hares['sex_simple'] == '?') | (hares['sex_simple'] == 'pf') , 'sex_simple'] = 'm?'

In [ ]:
# Check value counts in sex_simple column
hares['sex_simple'].value_counts(dropna=False)

sex_simple
f      1721
m      1249
NaN     352
m?       58
Name: count, dtype: int64

### 7. Calculate mean weight

Use groupby() to calculate the mean weight by sex using the new column. Compare your result with a team who handled the other codes differently. Did your choice change the result?

Write a full sentence explaining the results you obtained. Don’t forget to include units.

A news brief says: “Female snowshoe hares at Bonanza Creek are heavier than males.” With your team, list a few reasons why the data might not support this claim. Think about how the data was collected. Then pick one reason and write code to check it.

In [61]:
hares_valid = hares[hares['weight'] > 0]
mean_weight = hares_valid.groupby('sex_simple')['weight'].agg('mean')
mean_weight

sex_simple
f     1365.164792
m     1349.935542
m?     835.744681
Name: weight, dtype: float64

Confirmed female hares are heavier than confirmed male hares.